# Day 2 · Lab 2 — FastMCP Server: Auth + Rate Limiting + Client

## What you'll build

1. A **FastMCP server** exposing a `get_credit_score` tool
2. **API key authentication** — reject calls without a valid key
3. **Token-bucket rate limiting** — 10 calls per minute per key
4. A **client** that discovers and calls the tool over stdio
5. Integration into a LangGraph node

## Prerequisites

- Lab 1 completed
- `mcp` package installed in `(base)`

## Step 1 — Environment

In [ ]:
import os, sys, subprocess
from pathlib import Path

try:
    from dotenv import load_dotenv
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "python-dotenv"])
    from dotenv import load_dotenv

load_dotenv(Path("~/agentic-lab/.env").expanduser(), override=False)
for k in ("ANTHROPIC_API_KEY","OPENAI_API_KEY","LANGSMITH_API_KEY"):
    if os.environ.get(k) == "":
        del os.environ[k]

os.environ.setdefault("MCP_API_KEY", "demo-key-day2-lab")

print(f"✓ OPENROUTER_API_KEY set: {bool(os.environ.get('OPENROUTER_API_KEY'))}")
print(f"✓ MCP_API_KEY set:        {bool(os.environ.get('MCP_API_KEY'))}")

## Step 2 — Write the FastMCP server to disk

MCP servers run as separate processes. The client spawns the server over stdio.

In [ ]:
from pathlib import Path

SERVER_CODE = "\"\"\"Credit Bureau MCP server with API-key auth and per-key rate limiting.\"\"\"\nimport os\nimport time\nfrom collections import defaultdict, deque\nfrom mcp.server.fastmcp import FastMCP\nfrom pydantic import BaseModel\n\nmcp = FastMCP(\"credit-bureau\")\n\n_call_history = defaultdict(deque)\n\ndef _rate_limit_check(api_key: str, max_calls: int = 10, window_sec: int = 60):\n    now = time.time()\n    history = _call_history[api_key]\n    while history and history[0] < now - window_sec:\n        history.popleft()\n    if len(history) >= max_calls:\n        raise PermissionError(f\"Rate limit: {max_calls} calls/{window_sec}s exceeded\")\n    history.append(now)\n\ndef _auth_check(api_key: str):\n    expected = os.getenv(\"MCP_API_KEY\")\n    if not expected or api_key != expected:\n        raise PermissionError(\"Invalid API key\")\n\nclass BureauResponse(BaseModel):\n    score: int\n    tier: str\n    reason: str\n\n@mcp.tool()\ndef get_credit_score(applicant_id: str, api_key: str) -> dict:\n    \"\"\"Fetch a simulated credit score for the applicant.\n\n    Args:\n        applicant_id: Unique applicant identifier (e.g. APP-001)\n        api_key: MCP server API key for authentication\n    \"\"\"\n    _auth_check(api_key)\n    _rate_limit_check(api_key)\n\n    seed = sum(ord(c) for c in applicant_id) % 500\n    score = 350 + seed\n    tier = \"high\" if score >= 700 else \"medium\" if score >= 600 else \"low\"\n\n    return BureauResponse(\n        score=score,\n        tier=tier,\n        reason=f\"Simulated bureau lookup for {applicant_id}\"\n    ).model_dump()\n\nif __name__ == \"__main__\":\n    mcp.run()\n"

server_path = Path("bureau_mcp_server.py")
server_path.write_text(SERVER_CODE)
print(f"✓ Server written to {server_path.resolve()}")
print(f"  ({server_path.stat().st_size} bytes)")

## Step 3 — Verify the server file compiles

In [ ]:
import py_compile
try:
    py_compile.compile("bureau_mcp_server.py", doraise=True)
    print("✓ bureau_mcp_server.py compiles cleanly")
except py_compile.PyCompileError as e:
    print(f"✗ Compile error: {e}")

## Step 4 — Client: connect and discover tools

`list_tools()` = dynamic tool discovery. The LLM would use this to find available tools at runtime.

In [ ]:
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


async def discover_tools():
    params = StdioServerParameters(
        command=sys.executable,
        args=["bureau_mcp_server.py"],
        env={**os.environ},
    )
    async with stdio_client(params) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()
            tools = await session.list_tools()
            return tools


tools = asyncio.run(discover_tools())
print(f"Server exposes {len(tools.tools)} tool(s):")
for t in tools.tools:
    desc = (t.description or "").split(chr(10))[0]
    print(f"  - {t.name}: {desc[:80]}")

## Step 5 — Call the tool with a valid API key

In [ ]:
async def call_tool_ok():
    params = StdioServerParameters(
        command=sys.executable,
        args=["bureau_mcp_server.py"],
        env={**os.environ},
    )
    async with stdio_client(params) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()
            result = await session.call_tool(
                "get_credit_score",
                arguments={"applicant_id": "APP-001", "api_key": os.environ["MCP_API_KEY"]},
            )
            return result


result = asyncio.run(call_tool_ok())
print("Result content:")
for c in result.content:
    print(f"  {c}")

## Step 6 — Auth failure with wrong API key

In [ ]:
async def call_tool_bad_key():
    params = StdioServerParameters(
        command=sys.executable,
        args=["bureau_mcp_server.py"],
        env={**os.environ},
    )
    async with stdio_client(params) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()
            result = await session.call_tool(
                "get_credit_score",
                arguments={"applicant_id": "APP-001", "api_key": "wrong-key"},
            )
            return result


result = asyncio.run(call_tool_bad_key())
print("Expected auth failure:")
for c in result.content:
    print(f"  {c}")
print(f"\nisError: {result.isError}")

## Step 7 — Rate limit demo

Send 12 rapid calls; the last 2 fail with the rate-limit error.

In [ ]:
async def rate_limit_test():
    params = StdioServerParameters(
        command=sys.executable,
        args=["bureau_mcp_server.py"],
        env={**os.environ},
    )
    async with stdio_client(params) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()
            results = []
            for i in range(12):
                r_ = await session.call_tool(
                    "get_credit_score",
                    arguments={"applicant_id": f"APP-{i:03d}", "api_key": os.environ["MCP_API_KEY"]},
                )
                text = r_.content[0].text[:80] if r_.content else ""
                results.append((i, r_.isError, text))
            return results


results = asyncio.run(rate_limit_test())
for i, is_err, text in results:
    marker = "✗" if is_err else "✓"
    print(f"  [{i:2d}] {marker} {text[:70]}")

## Step 8 — LangGraph node backed by the MCP tool

In [ ]:
from typing import TypedDict


class SimpleState(TypedDict):
    applicant_id: str
    bureau_score: int
    bureau_tier: str


async def _call_bureau_mcp(applicant_id: str) -> dict:
    params = StdioServerParameters(
        command=sys.executable,
        args=["bureau_mcp_server.py"],
        env={**os.environ},
    )
    async with stdio_client(params) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()
            result = await session.call_tool(
                "get_credit_score",
                arguments={"applicant_id": applicant_id, "api_key": os.environ["MCP_API_KEY"]},
            )
            data = json.loads(result.content[0].text)
            return data


def bureau_via_mcp(state: SimpleState) -> dict:
    data = asyncio.run(_call_bureau_mcp(state["applicant_id"]))
    return {"bureau_score": data["score"], "bureau_tier": data["tier"]}


import json
out = bureau_via_mcp({"applicant_id": "APP-999"})
print(f"MCP-backed bureau node returned: {out}")

## What you learned

1. **FastMCP server structure** — `@mcp.tool()` decorator, Pydantic return types
2. **API key auth** — reject on missing/wrong key before the tool runs
3. **Token-bucket rate limiting** — protect API quotas from runaway agents
4. **Stdio client** — spawn server as subprocess, discover tools dynamically
5. **LangGraph integration** — wrap async MCP calls in sync node functions

## Production notes

- **Stdio is dev-only.** For prod, run the MCP server over HTTP behind a gateway.
- **API keys in `.env`.** Never hard-code.
- **Rate limits per API key**, not global — each tenant gets their own bucket.